[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/magrilu/cv-dojo/blob/main/notebooks/two-views/eight-point.ipynb)

# The eight-point algorithm

The previous notebook built the fundamental matrix out of two known cameras. Here
we do without them: only pairs of image points, and the constraint
$\mathbf{x}'^\top \mathsf{F}\mathbf{x} = 0$ that every correspondence must
satisfy.

That constraint is linear in the nine entries of $\mathsf{F}$, which is the whole
reason the problem is tractable at all — and, as we shall see, also the reason
the answer needs fixing afterwards.

In [ ]:
#| echo: false
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src")); break

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection

from cvdojo.house import (load_image, load_model, load_two_view_cameras,
                          load_annotation, project)
from cvdojo.plotting import (clip_line_to_image, draw_line_in_image,
                             pairwise_intersections, ACCENT)
from cvdojo.scene import skew, set_axes_equal

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY = "#3288BD", "0.45"

model = load_model()
V3 = {k: np.array(v, float) for k, v in model["vertices"].items()}
EDGES = model["edges"]
P, Pp = load_two_view_cameras()

I  = load_image("IMG_4331.jpeg")
Ip = load_image("IMG_4337.jpeg")
H_IMG, W_IMG = I.shape[:2]

IDS = list(V3)
X3 = np.array([V3[k] for k in IDS])
xs  = project(P,  X3)          # exact projections: our synthetic data
xps = project(Pp, X3)
h = lambda a: np.c_[np.atleast_2d(a), np.ones(len(np.atleast_2d(a)))]


# --- carried over from the fundamental matrix notebook ----------------------
C  = np.linalg.svd(P)[2][-1];  C  = C[:3]  / C[3]      # the camera centres
Cp = np.linalg.svd(Pp)[2][-1]; Cp = Cp[:3] / Cp[3]
X_h, Xp_h = h(xs), h(xps)

e_p = Pp @ np.append(C, 1.0)                 # the second epipole
F_geo = skew(e_p) @ Pp @ np.linalg.pinv(P)   # the fundamental matrix of this pair
F_geo = F_geo / np.linalg.norm(F_geo)


def epipoles(F):
    """e spans the right null space of F, e' the right null space of F^T."""
    e  = np.linalg.svd(F)[2][-1];   e  = e / e[2]
    ep = np.linalg.svd(F.T)[2][-1]; ep = ep / ep[2]
    return e, ep

## Computing $\mathsf{F}$ from correspondences

Everything so far assumed we knew the cameras. In practice we usually do not:
we have two photographs and a set of matched points. Even so, the epipolar
geometry can be estimated from the correspondences alone.

The whole algorithm, known as the **eight-point algorithm**, can be summarised
as follows.

1. Each correspondence gives one linear equation in the nine entries of
   $\mathsf{F}$.
2. Stack eight or more of them and take the null vector of the resulting matrix.
3. Do all of this in normalized coordinates, or the answer will be numerical
   noise.
4. Force the result to have rank two.

Step one first. Expanding $\mathbf{x}'^\top \mathsf{F} \mathbf{x} = 0$ with
$\mathbf{x} = (u, v, 1)$ and $\mathbf{x}' = (u', v', 1)$ gives

$$
\begin{bmatrix} u'u & u'v & u' & v'u & v'v & v' & u & v & 1\end{bmatrix}
\operatorname{vec}\mathsf{F} = 0,
$$

which is the Kronecker product $(\mathbf{x}' \otimes \mathbf{x})^\top
\operatorname{vec}\mathsf{F} = 0$: bilinear in the points, linear in the
unknowns.

How many do we need? $\mathsf{F}$ has nine entries, but it is defined only up to
scale — multiplying it by any non-zero constant leaves the constraint unchanged
— and it must have rank two. That is nine minus one minus one:

$$\text{7 degrees of freedom.}$$

Seven correspondences ought to be enough, and they are, but the rank condition
is cubic, so the solution is not unique and the problem stops being linear. At the moment, we prefer dropping the rank condition and treating $\mathsf{F}$
as an arbitrary matrix defined up to scale costs one more point and buys a
linear problem: hence we need **eight** points.


In [ ]:
def design_matrix(x, xp):
    """One row per correspondence: the Kronecker product x' (x) x."""
    u,  v  = x[:, 0],  x[:, 1]
    up, vp = xp[:, 0], xp[:, 1]
    return np.column_stack([up*u, up*v, up,
                            vp*u, vp*v, vp,
                            u,    v,    np.ones(len(x))])

A = design_matrix(X_h, Xp_h)
print("design matrix:", A.shape)
print("column magnitudes:", np.round(np.abs(A).max(axis=0), 1))


The entries of $\mathsf{A}$ span six
orders of magnitude: the $u'u$ column is of order $10^6$, the $u$ column of
order $10^3$, the last column is exactly $1$. We are about to ask for the
smallest singular vector of this matrix, and a matrix whose columns live on
wildly different scales has a condition number to match. The answer will be
dominated by rounding.

Hartley's fix is to change coordinates before solving: translate each set of
points so its centroid is at the origin, then scale so the average distance from
the origin is $\sqrt2$. Both are similarities, and a similarity of the image is
just a different choice of pixel units — the geometry does not care.


In [ ]:
#| echo: false
#| column: body
#| label: fig-normalisation
#| fig-cap: >-
#|   Hartley normalisation as a change of coordinates: precondition the points,
#|   solve the linear system where it is well conditioned, then carry the answer
#|   back to pixels.
fig, ax = plt.subplots(figsize=(7.2, 4.2), layout="constrained")
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
pos = {"p2": (1.4, 4.6), "p2s": (8.6, 4.6), "n2": (1.4, 1.2), "n2s": (8.6, 1.2)}
lab = {"p2": r"$\mathbb{P}^2$", "p2s": r"$\mathbb{P}^{2*}$",
       "n2": r"$\widehat{\mathbb{P}}^2$", "n2s": r"$\widehat{\mathbb{P}}^{2*}$"}
for k, (xx, yy) in pos.items():
    ax.text(xx, yy, lab[k], fontsize=17, ha="center", va="center")
arrows = [("p2", "p2s", r"$\mathsf{F}$", 0.35),
          ("n2", "n2s", r"$\widehat{\mathsf{F}}$", -0.55),
          ("p2", "n2", r"$\mathsf{T}$", 0.0),
          ("p2s", "n2s", r"$\mathsf{T}'^{-\top}$", 0.0)]
for a, b, t, off in arrows:
    (x0, y0), (x1, y1) = pos[a], pos[b]
    dx, dy = x1 - x0, y1 - y0
    n = np.hypot(dx, dy); ux, uy = dx / n, dy / n
    ax.annotate("", xy=(x1 - 0.7*ux, y1 - 0.45*uy), xytext=(x0 + 0.7*ux, y0 + 0.45*uy),
                arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.3))
    ax.text((x0+x1)/2 + (0 if dx else 0.45), (y0+y1)/2 + off,
            t, fontsize=14, color=ACCENT, ha="center", va="center")
ax.set_title("solve downstairs, then carry the answer back up", fontsize=11)
plt.show()


The diagram is the whole idea: we cannot solve well upstairs, so we push the
points down with $\mathsf{T}$ and $\mathsf{T}'$, solve there for
$\widehat{\mathsf{F}}$, and carry the result back with

$$\mathsf{F} = \mathsf{T}'^\top \widehat{\mathsf{F}}\, \mathsf{T}.$$

The transpose on $\mathsf{T}'$ is not a typo: lines transform contravariantly
to points, which is exactly what the right-hand arrow of the diagram says.


In [ ]:
def normalize_points(x):
    """Centroid to the origin, mean distance to the origin equal to sqrt(2)."""
    c = x[:, :2].mean(axis=0)
    d = np.linalg.norm(x[:, :2] - c, axis=1).mean()
    s = np.sqrt(2.0) / d
    T = np.array([[s, 0, -s*c[0]],
                  [0, s, -s*c[1]],
                  [0, 0,  1.0]])
    return (T @ x.T).T, T

xn, T = normalize_points(X_h)
print("before:  mean distance from the centroid =",
      f"{np.linalg.norm(X_h[:, :2] - X_h[:, :2].mean(0), axis=1).mean():8.1f} px")
print("after :  mean distance from the centroid =",
      f"{np.linalg.norm(xn[:, :2] - xn[:, :2].mean(0), axis=1).mean():8.4f}",
      "  (= sqrt(2))")
# the ninth singular value is the solution direction, so compare the eight
# that carry information: s1 / s8 is the conditioning that actually bites
s_raw = np.linalg.svd(design_matrix(X_h, Xp_h))[1]
s_nrm = np.linalg.svd(design_matrix(xn, normalize_points(Xp_h)[0]))[1]
print(f"spread of the informative singular values, s1/s8:")
print(f"   in pixels     {s_raw[0]/s_raw[7]:.1e}")
print(f"   normalized    {s_nrm[0]/s_nrm[7]:.1e}")


Six orders of magnitude of conditioning, recovered by a change of units.

One step remains. The null vector of $\mathsf{A}$ is some $3 \times 3$ matrix,
and on noisy data it will have full rank. Using the [Eckart–Young–Mirsky theorem](https://en.wikipedia.org/wiki/Low-rank_approximation),  we simply replace it by the closest rank-two matrix
in the Frobenius norm, which the singular value decomposition hands us:
compute $\widehat{\mathsf{F}} = \mathsf{U}\operatorname{diag}(\sigma_1, \sigma_2,
\sigma_3)\mathsf{V}^\top$ and set $\sigma_3$ to zero.


We also need a way to say how well an estimate fits the data. Above we measured
the distance of $\mathbf{x}'$ from the line $\mathsf{F}\mathbf{x}$; from here on
we average it with the distance of $\mathbf{x}$ from $\mathsf{F}^\top\mathbf{x}'$,
because $\mathsf{F}$ defines epipolar lines in both directions and on noisy data
there is no reason to measure the error in one image and not the other.

This is a choice, not a definition handed down. The algebraic residual
$\mathbf{x}'^\top\mathsf{F}\mathbf{x}$ is easier to compute but has no units and
no geometric meaning; the symmetric distance is in pixels, which is what we can
judge; and there is a third quantity, the Sampson distance, that approximates
the reprojection error better than either.

In [ ]:
def epipolar_distance(F, x, xp):
    """Symmetric point-to-epipolar-line distance, in pixels."""
    l  = (F @ x.T).T
    lp = (F.T @ xp.T).T
    d1 = np.abs(np.sum(xp * l,  axis=1)) / np.linalg.norm(l[:,  :2], axis=1)
    d2 = np.abs(np.sum(x  * lp, axis=1)) / np.linalg.norm(lp[:, :2], axis=1)
    return (d1 + d2) / 2

All in all the eight-point algorithm can be implemented as follows:

In [ ]:
def eight_point(x, xp, enforce_rank2=True):
    """Estimate F from n >= 8 correspondences, in homogeneous pixel coordinates."""
    # normalize the correspondences
    xn,  T  = normalize_points(x)
    xpn, Tp = normalize_points(xp)
    # build the design matrix
    A = design_matrix(xn, xpn)
    # solve a linear system to get F
    Fh = np.linalg.svd(A)[2][-1].reshape(3, 3)

    if enforce_rank2:
        U, s, Vt = np.linalg.svd(Fh)
        s[-1] = 0.0
        Fh = U @ np.diag(s) @ Vt

    # denormalize
    F = Tp.T @ Fh @ T
    return F / np.linalg.norm(F)



Time to try it on data. Since, we know the projections matrices, we can compare the fundamental matrix computed from the cameras, with the one obtained via the eight-point algorithm.

In [ ]:
F_est = eight_point(X_h, Xp_h)
sgn = np.sign((F_est * F_geo).sum())

print("residuals on the synthetic data:",
      f"{epipolar_distance(F_est, X_h, Xp_h).max():.2e} px")
print("largest entrywise difference from the true F:",
      f"{np.abs(sgn*F_est - F_geo).max():.2e}")


The small residuals show that we were able to recover the fundamental matrix up to machine precision. In practice, however, we have noise, and to improve the statistical efficiency of our estimation, instead of considering just 8 points, we can increase the number of matches. We just used ten
correspondences, and the extra two do no harm, the system becomes
overdetermined and the null vector becomes a least-squares fit, which is exactly
what we want when the data is noisy. More points, better estimate.

And a warning: eight points in
a *bad configuration* are worse than useless. Here is the same estimate run on
every subset of eight, sorted by how far it lands from the truth.


In [ ]:
import itertools
scores = []
for c in itertools.combinations(range(len(IDS)), 8):
    c = list(c)
    Fc = eight_point(X_h[c], Xp_h[c])
    err = np.abs(np.sign((Fc*F_geo).sum())*Fc - F_geo).max()
    scores.append((err, tuple(IDS[i] for i in c)))
scores.sort()
print("best  subset:", scores[0][1],  f"  error {scores[0][0]:.1e}")
print("worst subset:", scores[-1][1], f"  error {scores[-1][0]:.1e}")


In [ ]:
#| echo: false
#| column: page
#| label: fig-minimal-sets
#| fig-cap: >-
#|   Different subsets of eight correspondences might give different estimates.
def subset_diag(c):
    x, xp = X_h[list(c)], Xp_h[list(c)]
    xn, T = normalize_points(x); xpn, Tp = normalize_points(xp)
    A = design_matrix(xn, xpn)
    S = np.linalg.svd(A)[1]
    Fh = np.linalg.svd(A)[2][-1].reshape(3, 3)
    return S[-1] / S[0], np.linalg.svd(Fh)[1][-1] / np.linalg.svd(Fh)[1][0]

best_ids, worst_ids = scores[0][1], scores[-1][1]
fig = plt.figure(figsize=(13, 5.6), layout="constrained")
for j, (sel, tag, err) in enumerate([(best_ids, "a subset that works", scores[0][0]),
                                     (worst_ids, "a subset that does not", scores[-1][0])]):
    idx = [IDS.index(s) for s in sel]
    s8, s3 = subset_diag(idx)
    ax = fig.add_subplot(1, 2, j + 1, projection="3d")
    for a, b in EDGES:
        ax.plot(*zip(V3[a], V3[b]), color="0.78", lw=0.8)
    for k, Q in V3.items():
        on = k in sel
        ax.scatter(*Q, s=42 if on else 14,
                   color=ACCENT if on else "0.7", depthshade=False, zorder=5)
    d = model["dimensions_cm"]
    ax.set_box_aspect((d["long_side"], d["depth"], d["total_height"]))
    ax.view_init(elev=18, azim=-62); ax.set_axis_off()
    ax.set_title(f"{tag}\n{' '.join(sel)}\n"
                 f"error {err:.0e}    "
                 r"$\sigma_8/\sigma_1$ = " f"{s8:.0e}    "
                 r"$\sigma_3/\sigma_1$ of $\widehat{\mathsf{F}}$ = " f"{s3:.2f}",
                 fontsize=10)
plt.show()


Eight points are enough only when they are in _general position_.
Forty of the forty-five subsets return the true matrix to machine precision — the
data is exact, after all — and five fail outright. The five are not unlucky. For
them the design matrix has rank seven instead of eight, so the null space is
two-dimensional: a whole pencil of matrices satisfies all eight equations
*exactly*, and nothing in the data can choose between them. Adding decimal places
would not help, because the answer is not there to be found.

The problem is in the configuration itself. It is a prism, the two gables are congruent,
one the translate of the other,  and the five failing subsets are exactly those
that keep four vertices of one gable together with their four translates. Which
configurations fail, and why, will be investigated later...

### On images

Let us now use real correspondences
clicked by hand on the two photographs, ten of them, the six house corners and
the four corners of the door. Their mean distance from the true epipolar lines
is about 1.7 pixels, which is what careful annotation looks like.


In [ ]:
ann = load_annotation("two_view_matches")
xr  = h(np.array([m["x"]  for m in ann["matches"]]))
xpr = h(np.array([m["xp"] for m in ann["matches"]]))
rid = [m["id"] for m in ann["matches"]]

d_true = epipolar_distance(F_geo, xr, xpr)
for name, dd in zip(rid, d_true):
    print(f"  {name:3s}  {dd:5.2f} px from the true epipolar line")
print(f"\n  mean {d_true.mean():.2f}   median {np.median(d_true):.2f}   max {d_true.max():.2f}")


In [ ]:
#| echo: false
def _label(name, prime):
    kind, k = ("d", name[1:]) if name.startswith("D") else ("x", name[1:])
    return (rf"$\mathbf{{{kind}}}'_{{{k}}}$" if prime
            else rf"$\mathbf{{{kind}}}_{{{k}}}$")

fig, axes = plt.subplots(1, 2, figsize=(13, 8), layout="constrained")
for ax, im, pts, prime, ttl in [
        (axes[0], I,  xr,  False, ann["images"]["view"]),
        (axes[1], Ip, xpr, True,  ann["images"]["view_prime"])]:
    ax.imshow(im)
    ax.scatter(pts[:, 0], pts[:, 1], s=60, facecolors="none",
               edgecolors=ACCENT, lw=2, zorder=5)
    for name, q in zip(rid, pts):
        ax.text(q[0] + 16, q[1] - 16, _label(name, prime), fontsize=11,
                color=BLUE, bbox=dict(fc="white", alpha=0.7, ec="none", pad=1))
    ax.set_xlim(pts[:, 0].min() - 150, pts[:, 0].max() + 150)
    ax.set_ylim(pts[:, 1].max() + 150, pts[:, 1].min() - 150)
    ax.set_title(ttl, fontsize=10); ax.axis("off")
plt.show()


Now estimate $\mathsf{F}$ from these, twice: once letting the null vector stand
as it comes out of the linear system, once forcing rank two. The difference is
barely visible in the numbers and evident in the picture.


In [ ]:
F_free = eight_point(xr, xpr, enforce_rank2=False)
F_rank2 = eight_point(xr, xpr, enforce_rank2=True)

for tag, F in [("no rank constraint", F_free), ("rank two enforced", F_rank2)]:
    print(f"{tag:20s}  numerical rank {np.linalg.matrix_rank(F)}   "
          f"residuals: mean {epipolar_distance(F, xr, xpr).mean():5.2f} px")


In [ ]:
#| echo: false
# frame both panels identically, wide enough to hold the image and the
# place where the lines are heading
allpts = np.vstack([pairwise_intersections((F @ xr.T).T) for F in (F_free, F_rank2)])
ctr = np.median(allpts, axis=0)
x0 = min(0, ctr[0]) - 300; x1 = max(W_IMG, ctr[0]) + 300
y0 = min(700, ctr[1]) - 300; y1 = max(H_IMG, ctr[1]) + 300

fig, axes = plt.subplots(1, 2, figsize=(13, 8), layout="constrained")
for ax, F, ttl in [(axes[0], F_free,  "no rank constraint"),
                   (axes[1], F_rank2, "rank two enforced")]:
    ax.imshow(Ip)
    L = (F @ xr.T).T
    for l in L:
        seg = clip_line_to_image(l, W_IMG, H_IMG)
        if seg is None:
            continue
        # extend the drawn segment towards the meeting point
        d = seg[1] - seg[0]; d = d / np.linalg.norm(d)
        far = np.vstack([seg[0] - 2600*d, seg[1] + 2600*d])
        ax.plot(far[:, 0], far[:, 1], color=ACCENT, lw=0.9, alpha=0.85)
    ax.scatter(xpr[:, 0], xpr[:, 1], s=45, facecolors="none",
               edgecolors=BLUE, lw=1.6, zorder=5)
    pts = pairwise_intersections(L)
    c = np.median(pts, axis=0)
    sp = np.median(np.linalg.norm(pts - c, axis=1))
    ax.scatter(pts[:, 0], pts[:, 1], s=14, color=BLUE, alpha=0.6, zorder=6)
    ax.set_title(f"{ttl}\nmedian spread of the 45 intersections: {sp:.0f} px",
                 fontsize=10)
    ax.set_xlim(x0, x1); ax.set_ylim(y1, y0); ax.set_aspect("equal")
    ax.axis("off")
plt.show()


On the left the epipolar lines almost meet: the
small blue dots are the 45 pairwise intersections, scattered over a couple of
hundred pixels. There is no epipole, because a rank-three matrix has no null
vector, and a family of lines with no common point is not an epipolar pencil.

On the right they meet exactly, at a single point. Setting one singular value to
zero is what turns a matrix that fits the data into a matrix that describes the epipolar
geometry.

Note also that the roles of the two images are symmetric. So transposing the constraint
gives $\mathbf{x}^\top \mathsf{F}^\top \mathbf{x}' = 0$: the same matrix, read
backwards, maps points of the second image to lines in the first.


### Questions to leave open

**The constraint is necessary — is it sufficient?** If two points correspond
then $\mathbf{x}'^\top\mathsf{F}\mathbf{x} = 0$. But every point of the line
$\mathsf{F}\mathbf{x}$ satisfies the same equation, and there are infinitely
many of them. What does the constraint actually rule out, and what is left for a
matcher to do?

**We force the rank down to two — what stops it from falling to one?** Nothing,
in the algorithm as written. A rank-one $\mathsf{F}$ is degenerate in a way that
the SVD step will happily produce if the second singular value is also small.
When does that happen, and would you notice?

**Forcing rank two made the residuals worse. Should it not have made them
better?** Compare the numbers above: the unconstrained estimate fits the ten
points more closely than the rank-two one, and both fit them more closely than
the *true* $\mathsf{F}$. What is being minimized, and is it what we care about?

**The epipoles estimated from the photographs are nowhere near the true ones.**
Compute them and see. Ten points, annotated to two pixels, and a quantity that
moves by hundreds — which part of the geometry is so badly conditioned, and
what would you do about it?

**What happens when the camera moves straight ahead?** The baseline points along
the optical axis, so the epipole lands inside the picture and the epipolar lines
radiate out from it. Where exactly, and what does the pencil look like?

**Is every rank-two $3\times3$ matrix a fundamental matrix?** For calibrated
cameras the analogous object, the essential matrix, needs two equal singular
values and has only five degrees of freedom.

**For what motion is $\mathsf{F}$ skew-symmetric?** Then every epipolar line
corresponds to itself, and the two pictures are related in a way you can spot by
eye.

**Does $\mathsf{F}$ still exist if the two photographs were taken at different
zoom?** The answer is the reason one works with $\mathsf{F}$ at all.


## Further reading

- Longuet-Higgins, H. C. "A computer algorithm for reconstructing a scene from two projections", *Nature* 293, 1981. The original eight-point algorithm.
- Hartley, R. "In defense of the eight-point algorithm", *IEEE TPAMI* 19(6), 1997. Why normalizing the coordinates matters as much as it does.
- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. Chapter 9 for the epipolar geometry, Chapter 11 for the computation of $\mathsf{F}$, and Result 17.1 for the minors.
- Fusiello, A. *Visione Computazionale: tecniche di ricostruzione tridimensionale*, Franco Angeli, 2018. Chapter 5.

---

**Luca Magri** — Computer Vision Dojo
Code MIT · text and figures CC BY-NC-ND 4.0
<https://magrilu.github.io/cv-dojo/>
